# 02d. Aplicar Splink — mãe + endereço

Carrega `splink_model_mae_endereco.json` do
[`02c_treinar_splink_mae_endereco.ipynb`](02c_treinar_splink_mae_endereco.ipynb).
Mesmas 14 regras de blocking do 02b. Exporta
`splink_predictions_mae_endereco.parquet`.

Atribuição:
[`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb).


In [1]:
import json
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from IPython.display import display
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    PREDICT_NUM_CHUNKS_LEFT,
    PREDICT_NUM_CHUNKS_RIGHT,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_MAE_ENDERECO,
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    ensure_output_dir,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
SPLINK_MODEL_JSON = SPLINK_MODEL_MAE_ENDERECO
SPLINK_PREDICTIONS = SPLINK_PREDICTIONS_MAE_ENDERECO
if not SPLINK_MODEL_JSON.exists():
    raise RuntimeError(
        f'Modelo Splink não encontrado: {SPLINK_MODEL_JSON}. '
        'Rode notebooks/02c_treinar_splink_mae_endereco.ipynb — este notebook não retreina.'
    )

con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)
print('Modelo:', SPLINK_MODEL_JSON)
print('Predict chunks L/R:', PREDICT_NUM_CHUNKS_LEFT, PREDICT_NUM_CHUNKS_RIGHT)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

splink_input → censo_limpo_aplicacao ∪ cpf_limpo_aplicacao
Registros: 14,010,495
DuckDB: threads=20, memory_limit=344.5 GiB (defaults: 20, 370GB)
Modelo: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/splink_model_mae_endereco.json
Predict chunks L/R: None None


## Blocking de predição

Quatorze regras **OR** para gerar candidatos no `predict` (recall). Não são o
blocking do EM nem o do prior (esses ficam no 02).

`nome_meio` e `cpf_norm` ficam de fora. Quem não tem meio sumiria do predict;
CPF na predição faria a GT sempre candidata e o 03 circularia.
Município nas duas regras de primeiro nome (mês+dia e mês+ano). CEP nas duas
de último. Sexo em `ultimo+mes+dia+cep`, `ultimo+mes+ano+cep`, na 11ª
(último + mês/ano + primeiro DL) e nas três de idade. A 11ª não exige
primeiro exact: Damerau proporcional (`len >= 6` e `dl * 6 <= min(len)`).
Gênero `ANTONIO`/`ANTONIA` nessa regra cai no sexo. A 12ª é primeiro +
último + município + sexo, com idade a até 1 ano (Censo sem data). A 13ª não
usa o nome da pessoa: `logradouro_norm` + CEP + município + sexo, com a mesma
idade. A 14ª é `nome_mae_phon` + município + sexo, com a mesma idade: Censo
sem data quando o nome da pessoa não fecha. `nome_mae*` não entra no score.


In [2]:
from splink import block_on

blocking_rules = [
    block_on('nome_completo_phon'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'dia_nascimento', 'cod_municipio'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'ano_nascimento', 'cod_municipio'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento', 'sexo', 'cep'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento', 'sexo', 'cep'),
    (
        "l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.ano_nascimento = r.ano_nascimento "
        "AND l.mes_nascimento = r.mes_nascimento "
        "AND l.sexo = r.sexo "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) <= 2 "
        "AND least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon)) >= 6 "
        "AND damerau_levenshtein(l.primeiro_nome_phon, r.primeiro_nome_phon) * 6 "
        "<= least(len(l.primeiro_nome_phon), len(r.primeiro_nome_phon))"
    ),
    (
        "l.primeiro_nome_phon = r.primeiro_nome_phon "
        "AND l.ultimo_nome_phon = r.ultimo_nome_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.logradouro_norm = r.logradouro_norm "
        "AND l.cep = r.cep "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
    (
        "l.nome_mae_phon = r.nome_mae_phon "
        "AND l.cod_municipio = r.cod_municipio "
        "AND l.sexo = r.sexo "
        "AND abs(l.idade - r.idade) <= 1"
    ),
]


In [3]:
from splink.blocking_analysis import count_comparisons_from_blocking_rules
from splink.internals.charts import CumulativeBlockingRuleComparisonsGeneratedChart

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_chart = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_chart = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
print(f'Chart blocking (conjunto inteiro): censo {n_censo_chart:,} | cpf {n_cpf_chart:,}')

censo_in = db_api.register('splink_censo')
censo_in.dataset_display_name = 'censo'
cpf_in = db_api.register('splink_cpf')
cpf_in.dataset_display_name = 'cpf'

# 1.0 conta todos os pares. O default do Splink 5 amostra 5%.
contagem = count_comparisons_from_blocking_rules(
    [censo_in, cpf_in],
    blocking_rules=blocking_rules,
    link_type='link_only',
    unique_id_column_name='unique_id',
    record_sample_proportion=1.0,
)
n_pares_blocking = contagem[-1]['cumulative_comparison_count']
print(f'Pares OR: {n_pares_blocking:,}')
CumulativeBlockingRuleComparisonsGeneratedChart(contagem)


Chart blocking (conjunto inteiro): censo 6,079,030 | cpf 7,931,465


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pares OR: 599,218,332


SplinkChart (use .to_html() or .save() to export HTML)

Linker no JSON do 02 + as 14 regras acima. Views = tabelas de aplicação
(`censo_limpo_aplicacao` / `cpf_limpo_aplicacao`).



In [4]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_limpo = con.execute(
    f'SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA_APLICACAO}'
).fetchone()[0]
n_cpf_limpo = con.execute(
    f'SELECT COUNT(*) FROM {TABELA_CPF_LIMPA_APLICACAO}'
).fetchone()[0]
n_censo_view = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_view = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
assert n_censo_view == n_censo_limpo and n_cpf_view == n_cpf_limpo, (
    f'Linker não está no conjunto de aplicação: '
    f'censo {n_censo_view} vs aplicação {n_censo_limpo}; '
    f'cpf {n_cpf_view} vs aplicação {n_cpf_limpo}'
)
print('Linker: aplicação', f'{n_censo_view:,}', f'{n_cpf_view:,}')

with open(SPLINK_MODEL_JSON, 'r') as file:
    data = json.load(file)
freq_antiga = False
for comp in data['comparisons']:
    if comp.get('output_column_name') != 'nome_completo_phon':
        continue
    for nivel in comp['comparison_levels']:
        if nivel.get('tf_adjustment_column') != 'primeiro_nome_phon':
            continue
        nivel['tf_adjustment_column'] = 'nome_completo_phon'
        nivel.pop('disable_tf_exact_match_detection', None)
        freq_antiga = True
if freq_antiga:
    with open(SPLINK_MODEL_JSON, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=2)
        file.write('\n')
    print('JSON: frequência do exato agora é nome_completo_phon')
data['retain_intermediate_calculation_columns'] = False
data['retain_matching_columns'] = False
data['blocking_rules_to_generate_predictions'] = blocking_rules

censo_in = db_api.register('splink_censo')
censo_in.dataset_display_name = 'censo'
cpf_in = db_api.register('splink_cpf')
cpf_in.dataset_display_name = 'cpf'
linker = Linker([censo_in, cpf_in], data)


Linker: aplicação 6,079,030 7,931,465


`predict(0.05)` gera candidatos. O 03 mede cobertura em `p ≥ 0,5`; a faixa
`[0,25, 0,5)` vai para o [`04b_avaliar_nota_baixa.ipynb`](04b_avaliar_nota_baixa.ipynb);
`[0,05, 0,25)` para o [`04d_avaliar_faixa_005_025.ipynb`](04d_avaliar_faixa_005_025.ipynb).


In [5]:
predict_kwargs = {'threshold_match_probability': 0.05}
if PREDICT_NUM_CHUNKS_LEFT is not None:
    predict_kwargs['num_chunks_left'] = PREDICT_NUM_CHUNKS_LEFT
if PREDICT_NUM_CHUNKS_RIGHT is not None:
    predict_kwargs['num_chunks_right'] = PREDICT_NUM_CHUNKS_RIGHT

df_predict = linker.inference.predict(**predict_kwargs)
pred = df_predict.physical_name
print('predictions table:', pred)
print('pares:', con.execute(f'SELECT COUNT(*) FROM {pred}').fetchone()[0])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Blocking time: 279.41 seconds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Predict time (post-blocking): 670.20 seconds


predictions table: __splink__df_predict_ee48e48a0
pares: 6315410


Parquet estreito (ids + score). O largo do Splink só vive até o COPY.


In [6]:
ensure_output_dir()
cols = [r[0] for r in con.execute(f'DESCRIBE {pred}').fetchall()]
keep = [
    c for c in ('unique_id_l', 'unique_id_r', 'match_probability', 'match_weight')
    if c in cols
]
con.execute(
    f"COPY (SELECT {', '.join(keep)} FROM {pred}) "
    f"TO '{SPLINK_PREDICTIONS}' (FORMAT PARQUET, COMPRESSION ZSTD)"
)
drop_splink_temp_tables(con)
con.execute('CHECKPOINT')
print('Predictions:', SPLINK_PREDICTIONS)
print('colunas:', keep)


Removidas 6 tabelas/views residuais __splink__*


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Predictions: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/splink_predictions_mae_endereco.parquet
colunas: ['unique_id_l', 'unique_id_r', 'match_probability', 'match_weight']


Distribuição de score no parquet (sem rótulos). Discordar nome/DOB deve penalizar bits, não ≈ 0.


In [7]:
pred_parquet = f"read_parquet('{SPLINK_PREDICTIONS}')"
agg = [
    'COUNT(*) AS n',
    'MIN(match_probability) AS min_p',
    'AVG(match_probability) AS avg_p',
    'MAX(match_probability) AS max_p',
]
if 'match_weight' in keep:
    agg.extend(
        [
            'MIN(match_weight) AS min_w',
            'AVG(match_weight) AS avg_w',
            'MAX(match_weight) AS max_w',
        ]
    )
display(con.execute(f'SELECT {", ".join(agg)} FROM {pred_parquet}').df())

if 'match_weight' in keep:
    # DuckDB 1.x não tem width_bucket. 20 faixas de 3 bits em [-20, 40].
    display(
        con.execute(f'''
        SELECT
            faixa,
            CASE WHEN faixa <= 0 THEN NULL ELSE -20 + 3 * (faixa - 1) END AS peso_min,
            CASE WHEN faixa >= 21 THEN NULL ELSE -20 + 3 * faixa END AS peso_max,
            n_pares
        FROM (
            SELECT
                CASE
                    WHEN match_weight < -20 THEN 0
                    WHEN match_weight > 40 THEN 21
                    WHEN match_weight = 40 THEN 20
                    ELSE CAST(FLOOR((match_weight + 20) / 3.0) AS INTEGER) + 1
                END AS faixa,
                COUNT(*) AS n_pares
            FROM {pred_parquet}
            WHERE match_weight IS NOT NULL
            GROUP BY 1
        )
        ORDER BY 1
        ''').df()
    )


,n,min_p,avg_p,max_p,min_w,avg_w,max_w
0,6315410,0.050006,0.752965,1.0,-4.247741,12.534931,48.541544


,faixa,peso_min,peso_max,n_pares
0,6,-5,-2,1203850
1,7,-2,1,663491
2,8,1,4,283506
3,9,4,7,349647
4,10,7,10,395507
5,11,10,13,612340
6,12,13,16,338554
7,13,16,19,495778
8,14,19,22,454523
9,15,22,25,255138


In [8]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')
con.close()


modelo       ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/splink_model_mae_endereco.json
predictions  ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21/splink_predictions_mae_endereco.parquet
